# NLP: Neden ve Neden Şimdi — eşlik eden notebook

*Uygulamalı Doğal Dil İşleme — Token'lardan Ajanlara* kitabının 1. bölümü.
Önce bölümü okuyun: **[NLP: Neden ve Neden Şimdi](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/01-neden-nlp.html)**.

1\. bölüm ölçmekten çok tartışıyor. Sezgi kontrollerinin her biri, soruyu bir sayıyla
karara bağlayan ileri bir bölümü işaret ediyor. Bu notebook, bölümün üç merkezî
iddiasını kendi makinenizde, gerçek veri üzerinde izlemenizi sağlıyor:

1. **Bir yüzde, baseline'ı olmadan hiçbir şey ifade etmez** (1. sezgi kontrolü).
2. **Aynı problemi çözmenin üç yolu** esnekliği maliyet, gecikme ve öngörülebilirlikle
   takas ediyor (bölümdeki karşılaştırma tablosu).
3. **Uyuşma, güvenilirlik değildir** (3. sezgi kontrolü).

**Sayılar nasıl okunur.** Aşağıda basılan her şey *bu koşunun* ürettiği şey. Kitap aynı
niceliği bildiriyorsa, hücre kitabın değerini yanına yazıyor ve ikisinin tutup tutmadığını
söylüyor. Süreler makinenize bağlı ve kimseninkiyle birebir tutmayacak; onları büyüklük
mertebesi ve oran olarak okuyun.

**Dil hakkında bir not.** Anlatım ve kod yorumları Türkçe; kodun kendisi ve *basılan*
etiketler İngilizce. Bunun sebebi, Türkçe bölümlerin kendi `text` bloklarının da İngilizce
olması — böylece burada bastığınız tabloyla kitaptaki tablo yan yana okunabiliyor.
İngilizce notebook ile bu notebook arasındaki kod farkı da yalnızca yorumlardan ibaret
kalıyor.

**Süre.** İlk indirmeden (yaklaşık 25 MB) sonra bir dizüstü CPU'sunda bir dakikanın
altında. 3c kısmı, yerel bir `ollama` sunucusu çalışıyorsa bir dakika daha ekliyor;
çalışmıyorsa kendini temiz biçimde atlıyor.

## 1. Kurulum

İlk hücre bu koşunun kullandığı yorumlayıcı ve paket sürümlerini kaydediyor. Sizin
sayılarınız bu notebook'ta kayıtlı olanlardan farklıysa, önce bu hücreyi karşılaştırın.

In [1]:
import hashlib
import os
import platform
import sys
import time
from collections import Counter
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, cohen_kappa_score,
                             confusion_matrix, f1_score)
from sklearn.model_selection import train_test_split

SEED = 42

print(f"Python        {platform.python_version()}  ({platform.system()} {platform.machine()})")
for pkg in ["numpy", "pandas", "scikit-learn", "pyarrow", "requests"]:
    print(f"{pkg:<13} {version(pkg)}")

Python        3.12.13  (Darwin arm64)
numpy         2.5.2
pandas        3.0.5
scikit-learn  1.9.0
pyarrow       25.0.1
requests      2.34.2


### Derlem

Veri, [4. bölümün](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/04-karsilastirma.html) kullandığı veri: **olumlu ya da olumsuz
etiketli Türkçe ürün yorumları**, bir Türk e-ticaret sitesinden toplanmış 235.165 satır.
Kısa, gündelik ve redakte edilmemiş — ki asıl mesele de bu.

**Lisans.** Veri kümesini Fatih Barmanbay derledi; Hugging Face'te
`fthbrmnby/turkish_product_reviews` adıyla **CC BY-SA 4.0** altında dağıtılıyor. Bu
benzer-paylaşım lisansı, kitabın kendi CC BY 4.0'ından daha kısıtlayıcı; ondan türettiğiniz
bir şeyi yeniden dağıtmadan önce kontrol edin. 4. bölüm veri kümesi kartının *söylemediği*
şeyleri de not ediyor: hangi site kazındı, ne zaman kazındı ve olumlu/olumsuz etiketi nasıl
türetildi.

Aşağıdaki yükleme kodu 4. bölümün kodu, satır satır: sütunları yeniden adlandır, boşlukları
kırp, birebir tekrar eden metinleri at. Tek ekleme yerel bir önbellek; böylece notebook'u
yeniden çalıştırmak dosyayı tekrar indirmiyor.

**Neye bakacaksınız:** `this run` ve `book` sütunları. 4. bölüm 235.165 satır, tekilleştirme
sonrası 233.508 satır ve %93,7 olumlu bildirmişti.

In [2]:
URL = ("https://huggingface.co/datasets/fthbrmnby/turkish_product_reviews/"
       "resolve/main/data/train-00000-of-00001.parquet")
CACHE = Path("data/cache/turkish_product_reviews.parquet")

if not CACHE.exists():
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    response = requests.get(URL, timeout=300)
    response.raise_for_status()
    CACHE.write_bytes(response.content)
print(f"parquet file  {CACHE.stat().st_size:,} bytes, "
      f"sha256 {hashlib.sha256(CACHE.read_bytes()).hexdigest()[:12]}")


def vs_book(label, ours, book, fmt="{:,}"):
    """Bu koşunun değerini kitabınkinin yanına, kitabın bastığı hassasiyette basar."""
    ours_s, book_s = fmt.format(ours), fmt.format(book)
    verdict = "same as the book" if ours_s == book_s else "DIFFERS from the book"
    print(f"{label:<30} this run {ours_s:>9}   book {book_s:>9}   {verdict}")


# 4. bölümün yükleme kodu
raw = pd.read_parquet(CACHE)
df = raw.rename(columns={"sentence": "text", "sentiment": "label"})
df["text"] = df["text"].astype(str).str.strip()
df = df.drop_duplicates(subset="text")

print()
vs_book("rows as downloaded", len(raw), 235_165)
vs_book("after de-duplication", len(df), 233_508)
vs_book("duplicates removed", len(raw) - len(df), 1_657)
vs_book("label 1 (positive)", int((df["label"] == 1).sum()), 218_747)
vs_book("label 0 (negative)", int((df["label"] == 0).sum()), 14_761)
vs_book("positive share, %", 100 * (df["label"] == 1).mean(), 93.7, "{:.1f}")

parquet file  24,354,762 bytes, sha256 a5ea59cd4f78



rows as downloaded             this run   235,165   book   235,165   same as the book
after de-duplication           this run   233,508   book   233,508   same as the book
duplicates removed             this run     1,657   book     1,657   same as the book
label 1 (positive)             this run   218,747   book   218,747   same as the book
label 0 (negative)             this run    14,761   book    14,761   same as the book
positive share, %              this run      93.7   book      93.7   same as the book


## 2. Baseline'ı olmayan bir yüzde

1\. sezgi kontrolü şunu soruyor: *bir satıcının modeli sizin görevinizde %94 doğru — bu iyi
mi?* Bölümün cevabı, bilemeyeceğiniz; çünkü eksik olan sayı taban oran. Kazandırmak
istediği alışkanlık da şu: **çoğunluk sınıfının oranını sorun ve çıkarın.**

Önce bölme. 4. bölüm bunu, 80/20 bölünen, 42 tohumlu, katmanlı 50.000 yorumluk bir alt
örneklem diye tarif ediyor. 4. bölüm bu tarifin yetersiz tanımlı olduğu konusunda kendisi
uyarıyor: bir prosedür ailesini adlandırıyor ve ailenin üyeleri birbirini tutmuyor.
[12. bölüm](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/12-siniflandirma.html) 50.000 için tam çağrıyı — tek bir katmanlı
`train_test_split` — geri kazandı; hücre de o çağrıyı kullanıyor. 80/20 adımı ise 4.
bölümün düzyazısını bizim okuyuşumuz, kitabın bastığı bir kod değil; bu yüzden hücre sonucu
varsaymak yerine kitabın sayılarıyla karşılaştırıyor.

12\. bölümün öğüdü, çağrının yanında örneklemin bir parmak izini basmak; hücre onu da
yapıyor. En keskin parmak izi olan 4. bölüm vektörleştiricisinin öznitelik sayısı ise
3b kısmında geliyor.

In [3]:
# 12. bölümün geri kazandığı, 50.000 yorumluk alt örneklem çağrısı
sub, _ = train_test_split(df, train_size=50_000, stratify=df["label"], random_state=SEED)
# 80/20 bölme: 4. bölümün "katmanlı, 42 tohumlu" ifadesini bizim okuyuşumuz
train, test = train_test_split(sub, test_size=0.2, stratify=sub["label"], random_state=SEED)

print("subsample fingerprint", hashlib.sha256(",".join(map(str, sorted(sub.index))).encode()).hexdigest()[:12])
print()
vs_book("train size", len(train), 40_000)
vs_book("train positive", int((train["label"] == 1).sum()), 37_471)
vs_book("train negative", int((train["label"] == 0).sum()), 2_529)
vs_book("test size", len(test), 10_000)
vs_book("test positive", int((test["label"] == 1).sum()), 9_368)
vs_book("test negative", int((test["label"] == 0).sum()), 632)

y_test = test["label"].to_numpy()
y_train = train["label"].to_numpy()

subsample fingerprint 1e724ccf17fa

train size                     this run    40,000   book    40,000   same as the book
train positive                 this run    37,471   book    37,471   same as the book
train negative                 this run     2,529   book     2,529   same as the book
test size                      this run    10,000   book    10,000   same as the book
test positive                  this run     9,368   book     9,368   same as the book
test negative                  this run       632   book       632   same as the book


### Akla gelebilecek en tembel model

Her yoruma "olumlu" deyin, metne hiç bakmayın. Burada tanımlanan `score` yardımcısı bu
notebook'taki her yaklaşım için kullanılıyor; böylece hepsi aynı biçimde ölçülüyor:

- **accuracy**, doğru etiketlenen yorumların oranı;
- **F1(neg)**, olumsuz sınıftaki — yani şikâyetlerdeki — F1 skoru;
- **macro-F1**, iki sınıf F1'inin ağırlıksız ortalaması; böylece nadir sınıf, yaygın olanı
  kadar sayılıyor.

**Neye bakacaksınız:** doğruluk ile macro-F1 arasındaki uçurum ve karışıklık matrisi.

In [4]:
def score(y_true, y_pred):
    """Yüzde cinsinden doğruluk, macro-F1 ve olumsuz sınıf F1'i."""
    f1_neg, f1_pos = f1_score(y_true, y_pred, labels=[0, 1], average=None, zero_division=0)
    return {"accuracy": 100 * accuracy_score(y_true, y_pred),
            "macro-F1": 100 * (f1_neg + f1_pos) / 2,
            "F1(neg)": 100 * f1_neg}


def show_confusion(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    print(pd.DataFrame(cm, index=["actual neg", "actual pos"],
                       columns=["predicted neg", "predicted pos"]).to_string())


majority_pred = np.ones_like(y_test)
majority = score(y_test, majority_pred)

show_confusion(y_test, majority_pred)
print()
vs_book("accuracy", majority["accuracy"], 93.68, "{:.2f}")
vs_book("macro-F1", majority["macro-F1"], 48.37, "{:.2f}")
vs_book("F1(neg)", majority["F1(neg)"], 0.00, "{:.2f}")

            predicted neg  predicted pos
actual neg              0            632
actual pos              0           9368

accuracy                       this run     93.68   book     93.68   same as the book
macro-F1                       this run     48.37   book     48.37   same as the book
F1(neg)                        this run      0.00   book      0.00   same as the book


**Nasıl okunur.** Modelin girdisi de parametresi de yok; doğruluğu basitçe test
kümesindeki olumlu yorumların oranı. Böyle bir veride "%94 doğru" diyen bir satıcı, bir
sabitin yarım puan üstündeki bir sayıyı söylüyor demektir.

Macro-F1 bunu açığa çıkarıyor: olumsuz sınıfın F1'i sıfır oluyor ve sıfır ile olumlu
sınıfın F1'inin ortalaması 50'nin altına düşüyor. [4. bölümün](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/04-karsilastirma.html)
macro-F1'i manşet yapıp doğruluğu yalnızca ne kadar yanıltıcı olduğunu göstermek için
bildirmesinin sebebi bu; [12. bölümün](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/12-siniflandirma.html#sec-siniflandirma-metrikler)
metriklere derinlemesine dönmesinin de.

İki sayıyı da gözünüzün önünde tutun. Aşağıdaki her yaklaşım onlara karşı okunacak.

## 3. Aynı problemi çözmenin üç yolu

Bölümdeki tablo üç yaklaşımı karşılaştırıyor: kelimeleri sayıp bir sınıflandırıcı
eğitmek, ön eğitimli bir encoder'ı fine-tune etmek ve bir büyük dil modelini
prompt'lamak. Özeti şu: **yeni yaklaşımlar esneklik satın alıyor ve bedelini maliyet,
gecikme ve öngörülebilirlikle ödüyor.**

Bu kısım, bir dizüstü CPU'sunda dakikalar içinde çalışan üç yaklaşımı aynı test
yorumları üzerinde, aynı biçimde puanlayarak koşturuyor:

- **3a. Elle yazılmış bir kural.** Tablonun bir sütunu değil: bölümün zaman çizelgesindeki
  ilk çağ, insanların dili kestirmek yerine *yasalaştırdığı* dönem. Burada olmasının
  sebebi, çoğu insanın önce bunu denemesi.
- **3b. Klasik sütun:** TF-IDF öznitelikleri ve lojistik regresyon; tam olarak 4. bölümün
  kurduğu gibi.
- **3c. LLM sütunu:** yerel bir `ollama` sunucusu üzerinden küçük, açık ağırlıklı bir
  modeli prompt'lamak.

Fine-tune edilmiş encoder sütunu burada koşmuyor, çünkü böyle bir modeli eğitmek GPU işi.
[12. bölüm](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/12-siniflandirma.html) bu derlemde bir tane fine-tune ediyor ve batch'li
hâlde belge başına yaklaşık 2 ms ölçüyor; bölümün tablosundaki rakam da o.

### 3a. Elle yazdığınız bir kural

On iki olumsuz, on olumlu ifade; bir kez yazıldı ve test kümesi üzerinde ayarlanmadı.
Kararın biçimi [5. bölümdeki](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/05-veri-etiketleme.html) kaba etiketleyiciyle aynı: bir
yoruma ancak olumsuz bir ifade geçiyor *ve* olumlu bir ifade geçmiyorsa olumsuz de; aksi
hâlde çoğunluk sınıfına düş.

Bu, 5. bölümün sözlüğü **değil**. Oradaki 36 olumlu ve 37 olumsuz ifade içeriyor ve kitap
bunları basmıyor; dolayısıyla bu kuralın sayılarının 5. bölümünkilerle tutmasını beklemeyin.

İki ön işleme adımı 4. bölümden geliyor. `tr_lower` Türkçeye uygun küçültme yapıyor, çünkü
Python'un `str.lower()` metodu `I`'yı `i`'ye eşliyor; oysa Türkçede `ı` olmalı. Kural ayrıca
U+0307'yi (COMBINING DOT ABOVE) siliyor: 4. bölümün çalıştığı alt örneklemin yaklaşık
%8,4'ünde bulunan başıboş bir işaret. O adım olmadan `deği̇l` içeren bir yorum `değil`
ifadesiyle eşleşmezdi.

Eşleştirme alt dizge üzerinden, yani `iade` aynı zamanda `iadesi`yi de yakalıyor. Bu hem bir
özellik hem bir hata; 2. alıştırma hatayı bulmanızı istiyor.

In [5]:
TR_LOWER = str.maketrans({"I": "ı", "İ": "i"})


def tr_lower(s):
    return s.translate(TR_LOWER).lower()


NEG = ["kötü", "berbat", "rezalet", "değil", "beğenmedim", "memnun kalmadım",
       "tavsiye etmem", "tavsiye etmiyorum", "iade", "bozuk", "kalitesiz", "pişman"]
POS = ["güzel", "harika", "mükemmel", "süper", "memnunum", "teşekkür",
       "tavsiye ederim", "beğendim", "kaliteli", "başarılı"]


def rule_label(text):
    t = tr_lower(text).replace("\u0307", "")
    neg = any(p in t for p in NEG)
    pos = any(p in t for p in POS)
    return 0 if (neg and not pos) else 1


start = time.perf_counter()
rule_pred = np.array([rule_label(t) for t in test["text"]])
rule_us_per_doc = 1e6 * (time.perf_counter() - start) / len(test)
rule = score(y_test, rule_pred)

show_confusion(y_test, rule_pred)
print()
for k, v in rule.items():
    print(f"{k:<9} {v:6.2f}   (majority baseline {majority[k]:6.2f})")
print(f"latency   {rule_us_per_doc:.1f} µs per review, pure Python, one at a time")

            predicted neg  predicted pos
actual neg            200            432
actual pos            303           9065

accuracy   92.65   (majority baseline  93.68)
macro-F1   65.67   (majority baseline  48.37)
F1(neg)    35.24   (majority baseline   0.00)
latency   8.5 µs per review, pure Python, one at a time


**Neye bakacaksınız:** sabit modelinkinin *altında* bir doğruluk, onun epey üstünde bir
macro-F1. Kural bazı şikâyetleri buluyor ve bedelini, içinde tesadüfen olumsuz bir kelime
geçen olumlu yorumlardaki yanlış alarmlarla ödüyor.

Bölüm kuralların neden tıkandığını anlatıyor: *dilin devasa bir kuyruğu var*. Aşağıda,
olumsuz etiketli olup kuralın olumlu dediği birkaç yorum var. Her birini, başka bir şeyi
bozmadan yakalayacak ifadeyi yazmayı deneyin. Önce dikkatle okuyun: her kaçırma kuralın
suçu değil ve 12. bölüm bu derlemde en kendinden emin hataların çoğu zaman etiket hatası
olduğunu buluyor.

In [6]:
missed = test[(y_test == 0) & (rule_pred == 1)]
print(f"{len(missed)} of {int((y_test == 0).sum())} negative test reviews were labelled positive. Five of them:\n")
for text in missed.sample(5, random_state=SEED)["text"]:
    print("-", text[:160] + ("…" if len(text) > 160 else ""))

432 of 632 negative test reviews were labelled positive. Five of them:

- artıları:1-fiyatı uygun
- bazi arkadaslar yazmis kucuk falan diyede tam kestiremedim aldim ama baya baya kucuk.biraz daha buyuk olsaydi iyiydi ama bu boyutuyla cok kotu.zincir baya cansi…
- ürün güzel ancak duvara çok yakın bu nedenle tvnin arkasına takmanız gereken kablo bağlantılarını takmakta zorlanabilirsiniz. ayrıca tvyi askı aparatına asarken…
- ben pembesini kullanıyorum su kaynatmaya başlarken çok sesli kullandıkca ses artttı 3 4ay içinde rengi lekelenip beyazlaşmasya başladı uzun süre kullanılmayacağ…
- güneş altında kaldığından yaklaşık 2 sene içinde çürüdü plastiği kırılmaya başladı. gölge altında kullanılması belki ömrünü uzatır. ancak ipi sağlam ve güvenli …


### 3b. Kelimeleri saymak ve bir sınıflandırıcı eğitmek

Bu, 4. bölümün en iyi temsili: kelime unigram'ları ve bigram'ları üzerinde TF-IDF, sınıf
ağırlıklı bir lojistik regresyona veriliyor. Ayarlar 4. bölümün ayarları; `lowercase=False`
ve `preprocessor=tr_lower` dahil — bu ikisi scikit-learn'ün Python'un `str.lower()`'ını
çağırmasını engelliyor.

Bu hücre aynı zamanda notebook'taki en keskin yeniden üretim kontrolü. 4. bölüm, kendi
sayılarıyla bir şey karşılaştırmadan önce bölme boyutlarını *ve* bu vektörleştiricinin
ürettiği **101.659 özniteliği** tutturmanız gerektiğini söylüyor; 12. bölüm öznitelik
sayısının neden ikna edici olan olduğunu açıklıyor: 50.000'lik iki farklı örneklemin altı
haneli bir sayıda anlaşması için hiçbir sebep yok. 4. bölüm 92,57 doğruluk, 76,21 macro-F1
ve 56,47 F1(neg) basmıştı.

**Neye bakacaksınız:** öznitelik sayısı, üç skor ve eğitim süresi. 1. bölüm bu derlemde
yaklaşık bir saniyede eğitilen seyrek bir modelden söz ediyor (4. bölüm fit süresini 1,0 sn
ve ayrıca 0,9 sn ölçmüştü).

In [7]:
PAT = r"[a-zçğıöşü]+"
vec = TfidfVectorizer(min_df=2, sublinear_tf=True, ngram_range=(1, 2), token_pattern=PAT,
                      preprocessor=tr_lower, lowercase=False)
clf = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)

start = time.perf_counter()
X_train = vec.fit_transform(train["text"])
vectorise_s = time.perf_counter() - start
clf.fit(X_train, y_train)
train_s = time.perf_counter() - start

tfidf_pred = clf.predict(vec.transform(test["text"]))
tfidf = score(y_test, tfidf_pred)

show_confusion(y_test, tfidf_pred)
print()
vs_book("features", X_train.shape[1], 101_659)
vs_book("accuracy", tfidf["accuracy"], 92.57, "{:.2f}")
vs_book("macro-F1", tfidf["macro-F1"], 76.21, "{:.2f}")
vs_book("F1(neg)", tfidf["F1(neg)"], 56.47, "{:.2f}")
print(f"\ntraining time {train_s:.2f} s on 40,000 reviews "
      f"({vectorise_s:.2f} s building features, {train_s - vectorise_s:.2f} s fitting)")

            predicted neg  predicted pos
actual neg            482            150
actual pos            593           8775

features                       this run   101,659   book   101,659   same as the book
accuracy                       this run     92.57   book     92.57   same as the book
macro-F1                       this run     76.21   book     76.21   same as the book
F1(neg)                        this run     56.47   book     56.47   same as the book

training time 1.01 s on 40,000 reviews (0.91 s building features, 0.10 s fitting)


**Nasıl okunur.** Doğruluk yine sabit modelinkinin altında, macro-F1 ise hem sabitin hem
elle yazılmış kuralın epey üstünde. Burada kimse bir ifade listesi yazmadı; model 40.000
etiketli yorumdan kelime ve kelime çifti başına birer ağırlık kestirdi.

Bölümün tablosu klasik bir modelin **kararını tam olarak açıkladığını** iddia ediyor:
doğrusal bir skor, terim başına katkılara ayrışıyor. Sonraki hücre bunun ne demek olduğunu
gösteriyor. Önce her yöndeki en büyük ağırlıkları basıyor; sonra modelin kendinden emin
olduğu ilk olumsuz test yorumunu alıyor (skoru −2'nin altında, altı ile on iki arası etkin
terimle) ve terimlerini topluyor. Pozitif ağırlıklar "olumlu"ya itiyor; sabit terim artı
katkılar modelin skorunun *kendisi* ve hücre bunu kontrol ediyor.

In [8]:
terms = vec.get_feature_names_out()
weights = clf.coef_[0]                       # classes_ [0, 1]: pozitif ağırlık -> etiket 1
order = np.argsort(weights, kind="stable")
print("towards NEGATIVE                  towards POSITIVE")
for lo, hi in zip(order[:10], order[::-1][:10]):
    print(f"{terms[lo]:<22} {weights[lo]:+.2f}     {terms[hi]:<22} {weights[hi]:+.2f}")

# modelin kendinden emin olduğu ilk olumsuz yorum, bir avuç etkin terimle
X_test = vec.transform(test["text"])
nnz = np.diff(X_test.indptr)
test_scores = clf.decision_function(X_test)
i = next(j for j in range(len(test)) if y_test[j] == 0 and test_scores[j] < -2 and 6 <= nnz[j] <= 12)
row = X_test[i]
contrib = sorted(zip(terms[row.indices], row.data * weights[row.indices]), key=lambda t: t[1])

print(f"\nreview: {test['text'].iloc[i]}\n")
for term, c in contrib:
    print(f"  {term:<28} {c:+.3f}")
total = clf.intercept_[0] + sum(c for _, c in contrib)
print(f"  {'(intercept)':<28} {clf.intercept_[0]:+.3f}")
print(f"  {'score = sum of the above':<28} {total:+.3f}   "
      f"model's own decision_function {clf.decision_function(row)[0]:+.3f}  (< 0 means negative)")

towards NEGATIVE                  towards POSITIVE
değil                  -7.90     gayet                  +8.07
kötü                   -6.80     teşekkürler            +7.21
fakat                  -6.05     güzel                  +6.44
beğenmedim             -5.93     tavsiye ederim         +6.32
tavsiye etmiyorum      -5.83     ederim                 +6.30
etmiyorum              -5.58     mükemmel               +6.17
kalitesiz              -5.37     harika                 +6.06
iade                   -5.36     gerçekten              +5.69
ancak                  -4.94     memnunum               +5.47
ama                    -4.75     süper                  +4.99



review: çekiş gücü aşırı derecede düşük aranan performans yok

  düşük                        -0.950
  aşırı                        -0.706
  aşırı derecede               -0.369
  çekiş gücü                   -0.307
  gücü                         -0.274
  derecede                     -0.254
  çekiş                        -0.232
  yok                          -0.215
  aranan                       +0.115
  performans                   +0.510
  (intercept)                  +0.262
  score = sum of the above     -2.420   model's own decision_function -2.420  (< 0 means negative)


Şimdi tablonun gecikme satırı; klasik bir modelin **batch'li hâlde belge başına
mikrosaniyelerle** çalıştığını söylüyor. Hücre, 10.000 test yorumunun tamamında tahmini bir
kerede (beş koşunun medyanı), sonra da 500 yorum üzerinde teker teker ölçüyor.

Karşılaştırma için: [12. bölüm](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/12-siniflandirma.html) bu hattı başka bir makinede
batch'li 1.000 belge başına 0,014 sn, teker teker 1.000 belge başına 0,153 sn ölçtü; yani
belge başına 14 µs ve 153 µs.

In [9]:
batched = []
for _ in range(5):
    start = time.perf_counter()
    clf.predict(vec.transform(test["text"]))
    batched.append(time.perf_counter() - start)
tfidf_us_batched = 1e6 * np.median(batched) / len(test)

single = []
for text in test["text"].iloc[:500]:
    start = time.perf_counter()
    clf.predict(vec.transform([text]))
    single.append(time.perf_counter() - start)
tfidf_us_single = 1e6 * np.median(single)

print(f"batched       {tfidf_us_batched:7.1f} µs per review  (10,000 reviews, median of 5 runs)")
print(f"one at a time {tfidf_us_single:7.1f} µs per review  (median over 500 reviews)")

summary = pd.DataFrame(
    [["majority constant", *majority.values(), "none", "—"],
     ["hand-written rule", *rule.values(), "none (hand-written)", f"{rule_us_per_doc:.1f} µs"],
     ["tf-idf + logistic regression", *tfidf.values(), f"{train_s:.2f} s, 40,000 labels", f"{tfidf_us_batched:.1f} µs"]],
    columns=["approach", "accuracy", "macro-F1", "F1(neg)", "training", "latency per review"],
).set_index("approach")
print()
print(summary.round(2).to_string())

batched          18.6 µs per review  (10,000 reviews, median of 5 runs)
one at a time   148.9 µs per review  (median over 500 reviews)

                              accuracy  macro-F1  F1(neg)               training latency per review
approach                                                                                           
majority constant                93.68     48.37     0.00                   none                  —
hand-written rule                92.65     65.67    35.24    none (hand-written)             8.5 µs
tf-idf + logistic regression     92.57     76.21    56.47  1.01 s, 40,000 labels            18.6 µs


### 3c. Yerel bir LLM'i prompt'lamak

Tablonun son sütunu: hiç eğitim etiketi yok, yalnızca bir yönerge. Model `llama3.2:3b`;
yerel bir `ollama` sunucusunun servis ettiği, 3 milyar parametreli, açık ağırlıklı bir model.

**Korumalar.** [12. bölüm](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/12-siniflandirma.html#sec-siniflandirma-llm) tamamen bozuk
olduğu hâlde %93,75 doğruluk alan bir koşuyu anlatıyor: akıl yürüten bir model bütün token
bütçesini düşünmeye harcamış, her cevap boş dönmüş ve her boş cevap çoğunluk sınıfına
düşmüştü. Aşağıdaki düzenek bunun sessizce olamayacağı biçimde kuruldu:

- `temperature` 0 ve sabit bir `seed`, tek bir sabit prompt;
- `think: false` — 12. bölüm bunun bu modelde hiçbir şeyi değiştirmediğini, bozuk akıl
  yürütme koşusunu ise düzelttiğini bulmuştu; modeli değiştirirseniz diye burada duruyor;
- `num_predict` yalnızca bir üst sınır, bu yüzden her çağrı `done_reason`'ı ve varsa
  `thinking` metninin uzunluğunu kaydediyor;
- **ayrıştırılamayan bir cevap tahmin değildir.** Üçüncü bir sonuç olarak sayılıyor ve asla
  bir sınıfa eşlenmiyor. Skorlar, ayrıştırılan cevaplar üzerinden ve ayrıştırma oranıyla
  birlikte bildiriliyor;
- ilk 20 cevabın ayrıştırma oranı %50'nin altındaysa koşu durduruluyor;
- bir ısınma çağrısı modeli yüklüyor ve süreye dahil edilmiyor; bildirilen gecikme de
  ortalama değil medyan.

**Örneklem.** 50'si olumsuz, 50'si olumlu 100 test yorumu; 42 tohumuyla çekildi. Katmanlı
bir 100'lük örneklemde yalnızca altı kadar şikâyet olurdu ki bu macro-F1'in bir şey ifade
etmesi için fazla az. Denge baseline'ı değiştiriyor: bu örneklemde sabit model %93,68 değil
%50 doğruluk alıyor, dolayısıyla bu alt kısımdaki her sayıyı 50'ye karşı okuyun. 12.
bölümün kuralına uyarak kural ve TF-IDF modelleri de aynı 100 yorum üzerinde yeniden
puanlanıyor.

**Prompt İngilizce yazıldı.** 12. bölüm bu modelde İngilizce bir yönergenin Türkçe olanına
göre +16,90 macro-F1 değerinde olduğunu ölçtü. Kitap kendi prompt'unu basmıyor; aşağıdaki
yeni ve bu bir yeniden üretim değil, yeni bir ölçüm.

Hücre, sunucu cevap vermezse, model indirilmemişse ya da `ANLP_SKIP_LLM=1` ortam değişkeni
ayarlıysa kendini atlıyor.

In [10]:
OLLAMA_URL = os.environ.get("ANLP_OLLAMA_URL", "http://127.0.0.1:11434")
MODEL = "llama3.2:3b"
PROMPT = ("Classify the sentiment of this Turkish product review.\n"
          "Answer with exactly one word: positive or negative.\n\n"
          "Review: {review}\n\nAnswer:")

llm_ready = False
if os.environ.get("ANLP_SKIP_LLM") == "1":
    print("SKIPPED: ANLP_SKIP_LLM=1 is set. Section 3c will not run.")
else:
    try:
        server_version = requests.get(f"{OLLAMA_URL}/api/version", timeout=3).json()["version"]
        models = {m["name"]: m for m in requests.get(f"{OLLAMA_URL}/api/tags", timeout=3).json()["models"]}
        if MODEL in models:
            llm_ready = True
            m = models[MODEL]
            print(f"ollama {server_version} at {OLLAMA_URL}")
            print(f"model  {MODEL}  digest {m['digest'][:12]}  "
                  f"{m['details'].get('parameter_size', '?')} parameters, "
                  f"{m['details'].get('quantization_level', '?')}, {m['size'] / 1e9:.1f} GB")
        else:
            print(f"SKIPPED: ollama {server_version} is running but {MODEL} is not pulled "
                  f"(run `ollama pull {MODEL}`). Section 3c will not run.")
    except requests.RequestException as err:
        print(f"SKIPPED: no ollama server answered at {OLLAMA_URL} ({type(err).__name__}). "
              "Section 3c will not run; everything else in the notebook still does.")

ollama 0.24.0 at http://127.0.0.1:11434
model  llama3.2:3b  digest a80c4f17acd5  3.2B parameters, Q4_K_M, 2.0 GB


In [11]:
def ask(review):
    """Modele tek bir çağrı. Ham kaydı döndürür; ayrıştırma ayrıca yapılır."""
    payload = {"model": MODEL, "prompt": PROMPT.format(review=review), "stream": False,
               "think": False, "options": {"temperature": 0, "seed": SEED, "num_predict": 8}}
    start = time.perf_counter()
    r = requests.post(f"{OLLAMA_URL}/api/generate", json=payload, timeout=120)
    if r.status_code == 400 and "think" in r.text:     # alanı reddeden bir model
        payload.pop("think")
        r = requests.post(f"{OLLAMA_URL}/api/generate", json=payload, timeout=120)
    secs = time.perf_counter() - start
    r.raise_for_status()
    body = r.json()
    return {"raw": body.get("response", ""), "done_reason": body.get("done_reason"),
            "eval_count": body.get("eval_count"), "think_chars": len(body.get("thinking") or ""),
            "secs": secs}


def parse(raw):
    """'positive' -> 1, 'negative' -> 0, başka her şey -> None (ayrıştırılamaz, asla sınıf değil)."""
    word = raw.strip().strip(".!*\"'` \n").lower()
    return {"positive": 1, "negative": 0}.get(word)


llm_records = None
if llm_ready:
    sample = pd.concat([test[test["label"] == 0].sample(50, random_state=SEED),
                        test[test["label"] == 1].sample(50, random_state=SEED)])
    sample = sample.sample(frac=1, random_state=SEED)   # sınıfları zamana yay

    ask("Ürün güzel.")                                   # ısınma: modeli yükler, süreye girmez
    records = []
    for n, (idx, row) in enumerate(sample.iterrows(), start=1):
        rec = ask(row["text"])
        rec.update(index=idx, y=int(row["label"]), pred=parse(rec["raw"]))
        records.append(rec)
        if n == 20:
            rate = sum(r["pred"] is not None for r in records) / n
            if rate < 0.5:
                print(f"ABORTED after {n} answers: parse rate {rate:.0%}. "
                      "Inspect the raw answers below before trusting any score.")
                for r in records[:5]:
                    print(repr(r["raw"]), r["done_reason"], r["think_chars"])
                break
    else:
        llm_records = pd.DataFrame(records).set_index("index")

    if llm_records is not None:
        parsed = llm_records["pred"].notna()
        print(f"answers        {len(llm_records)}")
        print(f"parsed         {int(parsed.sum())}")
        print(f"unparseable    {int((~parsed).sum())}")
        print(f"done_reason    {dict(sorted(Counter(llm_records['done_reason']).items()))}")
        print(f"thinking chars {int(llm_records['think_chars'].sum())} in total")
        print(f"raw answers    {dict(Counter(llm_records['raw']).most_common())}")
        print(f"latency        median {llm_records['secs'].median():.3f} s per review "
              f"(range {llm_records['secs'].min():.3f}–{llm_records['secs'].max():.3f} s)")
        if (~parsed).any():
            print("\nunparseable answers:", llm_records.loc[~parsed, "raw"].tolist())
else:
    print("SKIPPED: the model was not called (see the cell above).")

answers        100
parsed         100
unparseable    0
done_reason    {'stop': 100}
thinking chars 0 in total
raw answers    {'positive.': 28, 'Negative.': 26, 'positive': 22, 'Negative': 15, 'negative': 5, 'negative.': 3, 'Positive.': 1}
latency        median 0.107 s per review (range 0.094–0.172 s)


**Aynı yorumlar üzerinde puanlama.** Sonraki hücre dört yaklaşımı da LLM'in cevapladığı
yorumlar üzerine koyuyor: ayrıştırılamayan cevap yoksa 100'ünün tamamı. Macro-F1'in
yanındaki aralık, bu yorumlar üzerinde %95'lik bir yüzdelik bootstrap (2.000 yeniden
örnekleme). Orada olmasının sebebi, 100 yorumun küçük farkları ayırt edememesi; aralığın
genişliği de bunu söylemenin dürüst yolu.

In [12]:
def bootstrap_macro_f1(y_true, y_pred, n_boot=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    stats = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        stats.append(score(y_true[idx], y_pred[idx])["macro-F1"])
    return np.percentile(stats, [2.5, 97.5])


if llm_records is None:
    print("SKIPPED: no LLM answers to score (see the cells above).")
else:
    answered = llm_records[llm_records["pred"].notna()]
    y_s = answered["y"].to_numpy()
    positions = test.index.get_indexer(answered.index)
    preds = {
        "majority constant": np.ones_like(y_s),
        "hand-written rule": rule_pred[positions],
        "tf-idf + logistic regression": tfidf_pred[positions],
        f"{MODEL}, zero-shot": answered["pred"].astype(int).to_numpy(),
    }
    latency = {
        "majority constant": "—",
        "hand-written rule": f"{rule_us_per_doc / 1e3:.4f} ms",
        "tf-idf + logistic regression": f"{tfidf_us_batched / 1e3:.4f} ms",
        f"{MODEL}, zero-shot": f"{1e3 * answered['secs'].median():.1f} ms",
    }
    rows = []
    for name, p in preds.items():
        s = score(y_s, p)
        lo, hi = bootstrap_macro_f1(y_s, p)
        rows.append([name, s["accuracy"], s["macro-F1"], f"[{lo:.1f}, {hi:.1f}]", s["F1(neg)"], latency[name]])
    table = pd.DataFrame(rows, columns=["approach", "accuracy", "macro-F1", "95% CI",
                                        "F1(neg)", "latency per review"]).set_index("approach")
    print(f"{len(y_s)} reviews: {int((y_s == 0).sum())} negative, {int((y_s == 1).sum())} positive\n")
    print(table.round(2).to_string())
    print(f"\nthe LLM's median latency is {1e3 * answered['secs'].median() / (tfidf_us_batched / 1e3):,.0f}x "
          "the tf-idf model's batched latency per review")
    print(f"\n{MODEL} confusion matrix:")
    show_confusion(y_s, preds[f"{MODEL}, zero-shot"])

    # dengeli örneklemin gizlediği: derlemin kendi oranında şikâyet precision'ı
    prevalence = (y_test == 0).mean()
    measured_on_full = {"hand-written rule": rule_pred, "tf-idf + logistic regression": tfidf_pred}
    print(f"\ncomplaint precision at the full test set's {100 * prevalence:.2f}% complaint rate")
    print(f"{'':<30}{'recall(neg)':>12}{'false alarms':>14}{'projected':>11}{'measured, n=10,000':>20}")
    for name, p in list(preds.items())[1:]:
        recall = (p[y_s == 0] == 0).mean()
        false_alarm = (p[y_s == 1] == 0).mean()
        denom = recall * prevalence + false_alarm * (1 - prevalence)
        projected = f"{100 * recall * prevalence / denom:.1f}" if denom > 0 else "—"
        if name in measured_on_full:
            full_pred = measured_on_full[name]
            measured = f"{100 * ((full_pred == 0) & (y_test == 0)).sum() / (full_pred == 0).sum():.1f}"
        else:
            measured = "not run"
        print(f"{name:<30}{100 * recall:>11.1f}%{100 * false_alarm:>13.1f}%{projected:>11}{measured:>20}")

100 reviews: 50 negative, 50 positive

                              accuracy  macro-F1        95% CI  F1(neg) latency per review
approach                                                                                  
majority constant                 50.0     33.33  [28.6, 37.5]     0.00                  —
hand-written rule                 61.0     54.00  [43.4, 63.8]    36.07          0.0085 ms
tf-idf + logistic regression      86.0     85.91  [78.6, 92.8]    84.78          0.0186 ms
llama3.2:3b, zero-shot            89.0     89.00  [82.4, 95.0]    88.89           107.0 ms

the LLM's median latency is 5,762x the tf-idf model's batched latency per review

llama3.2:3b confusion matrix:
            predicted neg  predicted pos
actual neg             44              6
actual pos              5             45

complaint precision at the full test set's 6.32% complaint rate
                               recall(neg)  false alarms  projected  measured, n=10,000
hand-written rule         

**Nasıl okunur.** Herhangi bir skordan önce ayrıştırma sayısına bakın: ayrışmayan
cevaplar üzerinden alınan bir skor, skor değildir. Sonra dört satırı bu örneklemin %50'lik
baseline'ına karşı okuyun ve aralıkların ne kadar geniş olduğuna bakın. 100 yorumla,
aralıkları büyük ölçüde örtüşen satırlar birbirinden ayrılmış sayılmaz. 12. bölüm tam bu
yüzden 2.000 belge ve *eşleştirilmiş* bir bootstrap kullanıyor.

**Dengeli örneklemin gizlediği şey.** Bu 100 yorumun yarısı şikâyet; derlemde ise olumlu
sınıf 14,8 kat daha büyük. Olumlu bir yorumdaki yanlış alarm bu örneklemde az, gerçek
trafikte çok pahalı. Son çıktı, her yaklaşımın şikâyet precision'ını örneklemdeki recall ve
yanlış alarm oranından yola çıkarak tam test kümesinin oranına izdüşürüyor. Kural ve TF-IDF
için izdüşümü, 10.000 yorumun tamamında ölçülen precision'la karşılaştırabilirsiniz. 50
olumlu yorumla yanlış alarm oranı 2 puanlık adımlarla hareket ettiği için izdüşümü kaba
okuyun: sıfır yanlış alarm, 50 yorumun destekleyemeyeceği kusursuz bir precision'a
izdüşüyor ve ölçülen sütun 10.000 yorumun ne dediğini gösteriyor.

**Bağlam, yeniden üretim değil.** 12. bölüm `llama3.2:3b`'yi kendi İngilizce prompt'uyla,
katmanlı 2.000 yorumluk bir alt örneklemde 69,21 macro-F1, %31,56 şikâyet precision'ı ve
belge başına 0,124 sn (medyan) ölçtü. Farklı prompt, farklı örneklem, farklı sınıf dengesi:
yukarıdaki sayılar onunla karşılaştırılabilir değil.

Bölümün tablosunun yaptığı vurgu son sütunda. Prompt'lanan modelin hiç eğitim etiketine
ihtiyacı olmadı ve yorum başına, yaklaşık bir saniyede eğitilen bir modelden büyüklük
mertebeleri daha fazla zamana mal oluyor. Yukarıda basılan oran, makineler arasında taşınan
kısım. Bölümün anlattığı takas da bu: **esneklik; bedeli maliyet, gecikme ve
öngörülebilirlik.** [12. bölüm](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/12-siniflandirma.html#sec-siniflandirma-kesisim) bu
takasın nerede tersine döndüğünü ölçüyor.

## 4. Uyuşma, güvenilirlik değildir

3\. sezgi kontrolü şunu soruyor: *iki etiketleyici etiketlerinizin %93'ünde hemfikir —
etiketler güvenilir mi?* Zorunlu olarak değil. Tek bir etiket baskın olduğunda, hiçbir şey
okumayan iki etiketleyici de çoğu zaman hemfikir çıkar. **Cohen'in kappa'sı**, ikisinin
kendi etiket oranlarıyla bağımsız etiketleseydi ulaşacağı uyuşmayı çıkarıyor:

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

Burada $p_o$ gözlenen uyuşma, $p_e$ ise iki etiketleyici her biri kendi frekanslarıyla
bağımsız etiketleseydi beklenen uyuşma.

### 4a. Kitabın sayılarını kendi tablosundan yeniden üretmek

[5. bölüm](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/05-veri-etiketleme.html) referans etiketleri (A etiketleyicisi) ile kaba bir
sözlük kuralını (B etiketleyicisi) 10.000 test yorumu üzerinde karşılaştırıyor ve
karışıklık tablosunu basıyor. Hücre $p_o$, $p_e$ ve kappa'yı yalnızca o dört sayıdan
yeniden kuruyor, sonra sonucu scikit-learn'ün `cohen_kappa_score`'uyla karşılaştırıyor.
Kitap $p_o$ = 0,9342, $p_e$ = 0,9099 ve kappa = 0,2698 veriyor; A öğelerin %6,32'sine,
B ise %3,08'ine olumsuz diyor.

In [13]:
def agreement(table):
    """2x2 tablodan p_o, p_e ve Cohen kappa: satırlar A, sütunlar B etiketleyicisi."""
    table = np.asarray(table, dtype=float)
    n = table.sum()
    p_o = np.trace(table) / n
    a_rates = table.sum(axis=1) / n          # A her etiketi ne sıklıkta kullanıyor
    b_rates = table.sum(axis=0) / n          # B her etiketi ne sıklıkta kullanıyor
    p_e = float(a_rates @ b_rates)
    return {"p_o": p_o, "p_e": p_e, "kappa": (p_o - p_e) / (1 - p_e),
            "A negative": a_rates[0], "B negative": b_rates[0]}


# 5. bölüm: A (referans, satırlar) ve B (sözlük kuralı, sütunlar), önce olumsuz
ch05_table = [[141, 491],
              [167, 9201]]
ch05 = agreement(ch05_table)

vs_book("A calls negative, %", 100 * ch05["A negative"], 6.32, "{:.2f}")
vs_book("B calls negative, %", 100 * ch05["B negative"], 3.08, "{:.2f}")
vs_book("p_o", ch05["p_o"], 0.9342, "{:.4f}")
vs_book("p_e", ch05["p_e"], 0.9099, "{:.4f}")
vs_book("kappa", ch05["kappa"], 0.2698, "{:.4f}")

# aynı kappa, tablodan açılan etiket vektörlerinden
a = np.repeat([0, 0, 1, 1], [141, 491, 167, 9201])
b = np.repeat([0, 1, 0, 1], [141, 491, 167, 9201])
print(f"\nscikit-learn cohen_kappa_score on the expanded table: {cohen_kappa_score(a, b):.4f}")
print(f"above the baseline: {100 * (ch05['p_o'] - ch05['p_e']):.2f} points, "
      f"out of {100 * (1 - ch05['p_e']):.2f} available")

A calls negative, %            this run      6.32   book      6.32   same as the book
B calls negative, %            this run      3.08   book      3.08   same as the book
p_o                            this run    0.9342   book    0.9342   same as the book
p_e                            this run    0.9099   book    0.9099   same as the book
kappa                          this run    0.2698   book    0.2698   same as the book

scikit-learn cohen_kappa_score on the expanded table: 0.2698
above the baseline: 2.43 points, out of 9.01 available


**Nasıl okunur.** 1. bölüm bu sayıları bir ayrıştırma olarak değil, bir karşılaştırma
olarak okuyor: 93,42 puanlık gözlenen uyuşmaya karşı, bu oranlara sahip iki etiketleyicinin
bağımsız etiketleyerek ulaşacağı 90,99. Geriye baseline'ın üstünde 2,43 puan kalıyor ve
mevcut olan yalnızca 9,01; kappa da bu boşluğun alınan payı: yaklaşık 0,27.

$p_e$, bir bağımsızlık sıfır hipotezinin öngördüğü şey. 90,99 puanın *hangisinin* şans
olduğunu söylemiyor.

### 4b. Tahmin yürüten iki doktor

Bölümün benzetmesi: bir taramaya gelen hastaların %95'i sağlıklı. İki doktor hiçbir dosyaya
bakmıyor ve her biri bağımsız olarak zamanın %95'inde "sağlıklı" diyor. Aynı kararda
buluşma oranları $0{,}95^2 + 0{,}05^2 = \%90{,}5$.

Hücre bu durumu ve birkaç farklı tahmin oranını 100.000 hasta üzerinde benzetiyor.
**Neye bakacaksınız:** tahmin oranı daha dengesiz hâle geldikçe ham uyuşma tırmanıyor, ama
kappa her seferinde yaklaşık sıfırda kalıyor — çünkü iki doktor da hiçbir bilgi
kullanmıyor.

In [14]:
rng = np.random.default_rng(SEED)
n_patients = 100_000
rows = []
for rate in [0.50, 0.80, 0.95, 0.99]:
    doctor_1 = (rng.random(n_patients) < rate).astype(int)   # 1 = "sağlıklı"
    doctor_2 = (rng.random(n_patients) < rate).astype(int)
    rows.append([rate, 100 * (rate**2 + (1 - rate)**2), 100 * (doctor_1 == doctor_2).mean(),
                 cohen_kappa_score(doctor_1, doctor_2)])
doctors = pd.DataFrame(rows, columns=["says healthy", "agreement, formula %",
                                      "agreement, simulated %", "kappa"])
print(doctors.round({"agreement, formula %": 2, "agreement, simulated %": 2, "kappa": 4}).to_string(index=False))

 says healthy  agreement, formula %  agreement, simulated %   kappa
         0.50                 50.00                   50.28  0.0055
         0.80                 68.00                   67.93 -0.0029
         0.95                 90.50                   90.53 -0.0007
         0.99                 98.02                   97.97 -0.0034


### 4c. Kendi etiketleyicileriniz

Şimdi aynı aritmetik, bu notebook'un kendi etiketleri üzerinde. Referans sütununu A
etiketleyicisi, 3. kısımdaki her yaklaşımı da B etiketleyicisi sayın; 10.000 test yorumunun
tamamında.

5\. bölümün kendi sürümüne iliştirdiği çekince burada da geçerli. Bu etiketleyicilerin
hiçbiri ikinci bir insan değil; dolayısıyla buradaki kappa, *her yaklaşımın referans
etiketlere ne kadar yaklaştığını* ölçüyor, etiketleme görevinin iyi tanımlanmış olup
olmadığını değil. Aritmetik her iki durumda da aynı.

**Neye bakacaksınız:** üç satırı önce ham uyuşmaya, sonra kappa'ya göre sıralayın.

In [15]:
rows = []
for name, pred in [("majority constant", majority_pred),
                   ("hand-written rule", rule_pred),
                   ("tf-idf + logistic regression", tfidf_pred)]:
    tbl = confusion_matrix(y_test, pred, labels=[0, 1])
    ag = agreement(tbl)
    assert abs(ag["kappa"] - cohen_kappa_score(y_test, pred)) < 1e-12
    rows.append([name, 100 * ag["p_o"], 100 * ag["p_e"], ag["kappa"], 100 * ag["B negative"]])

print("A (gold, rows) vs B (hand-written rule, columns), n = 10,000")
print(pd.DataFrame(confusion_matrix(y_test, rule_pred, labels=[0, 1]),
                   index=["A: negative", "A: positive"],
                   columns=["B: negative", "B: positive"]).to_string())
print()
kappas = pd.DataFrame(rows, columns=["annotator B", "raw agreement %", "p_e %", "kappa",
                                     "B calls negative %"]).set_index("annotator B")
print(kappas.round({"raw agreement %": 2, "p_e %": 2, "kappa": 4, "B calls negative %": 2}).to_string())

A (gold, rows) vs B (hand-written rule, columns), n = 10,000
             B: negative  B: positive
A: negative          200          432
A: positive          303         9065

                              raw agreement %  p_e %   kappa  B calls negative %
annotator B                                                                     
majority constant                       93.68  93.68  0.0000                0.00
hand-written rule                       92.65  89.29  0.3140                5.03
tf-idf + logistic regression            92.57  84.29  0.5271               10.75


**Nasıl okunur.** Referans etiketlerle ham uyuşma, doğruluğun *kendisi*; bu yüzden ham
uyuşmaya göre sıralama sabiti başa koyuyor: hiçbir şey okumadan referansla en sık o
uyuşuyor. Kappa'sı tam olarak sıfır, çünkü $p_e$'si $p_o$'suna eşit. Bu koşuda kappa bütün
sırayı tersine çeviriyor: sabit sona düşüyor ve ham uyuşması *en düşük* satır olan TF-IDF
başa geçiyor.

Bu, bölümün genel dersinin başka bir yönden gelişi: **bir yüzde, karşısında okunması
gereken baseline olmadan anlamsızdır.** 5. bölüm yanında taşınmaya değer iki uyarı daha
ekliyor. Sınıflar bu kadar çarpıkken kappa oynak olur, çünkü hücreler arasında yer
değiştiren bir avuç öğe küçük paydasını epey sallar. Ve iki etiketleyici azınlık etiketini
çok farklı oranlarda kullandığında kappa şişer. Tek bir katsayının yanına karışıklık
tablosunu da koyun.

## 5. Alıştırmalar

Bunlar [1. bölümün kendi alıştırmalarını](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/01-neden-nlp.html) genişletiyor ve daha
derine inen bölümleri işaret ediyor.

1. **Kendi baseline'ınız.** 1. bölümün ilk alıştırması kendi işinizden bir belge türü seçip
   "başarı"nın bir sayı olarak ne demek olacağını söylemenizi istiyor. Elinizde ona ait
   etiketli örnek varsa, her şeyden önce çoğunluk sınıfının oranını hesaplayın; sonra bir
   modelin bir sabiti geçmiş sayılması için gereken doğruluğu yazın. 1. bölümün ikinci
   mülakat sorusu işin sırasını veriyor; [4. bölüm](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/04-karsilastirma.html) ise ucuz
   modelin ne skor aldığını anlatan koca bir bölüm.

2. **Kuyruk, elle.** 3a kısmındaki `NEG` listesine `"sorun"` ekleyip kısmı yeniden
   koşturun. İçinde `sorunsuz` geçen yorumlara ne olduğuna bakın. Onarın, sonra kaç kural
   yazdığınızı ve hâlâ kaç şikâyetin kaçtığını sayın. Kural çağının başarısızlığı,
   küçültülmüş hâliyle budur.
   [2. bölüm](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/02-metin-veri.html#sec-metin-veri-zipf) sözlüğün neden hiç kapanmadığını
   anlatıyor; 4. bölüm de bu derlemin eğitim diliminde `kulla` önekini paylaşan 895 farklı
   kelime biçimi sayıyor.

3. **Klasik sütunun kaç etikete ihtiyacı var?** Bölümün tablosu "binlerce" diyor. 3b
   kısmını `train`'in 200, 1.000 ve 5.000 yorumluk katmanlı alt kümeleri üzerinde yeniden
   eğitip her birini tam test kümesinde puanlayın. Eğrinizi önce
   [12. bölümün baseline eğrisiyle](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/12-siniflandirma.html#sec-siniflandirma-baseline),
   sonra sınadığınız tablo satırıyla karşılaştırın.

4. **Prompt'ta tek bir şeyi değiştirin.** 3c kısmındaki `PROMPT`'u Türkçe olarak yeniden
   yazıp koşturun. 12. bölüm bu modelde İngilizce bir yönergeyi +16,90 macro-F1 değerinde
   ölçmüştü; sizin koşunuz aynı yönü mü gösteriyor ve 100 yorum bunu söyleyebilir mi? Sonra
   örneklemi 300'e çıkarıp aralıkların daralmasını izleyin.
   [17. bölüm](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/17-promptlama.html) prompt'lamayı,
   [18. bölüm](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/18-cozumleme.html) ise bölümün tablosunun prompt'lanan çıktı için
   gerekli dediği çıktı kısıtlamayı anlatıyor.

5. **Aynı etiketleyiciler, farklı örneklem.** 5. bölümün beşinci alıştırması: elle yazılmış
   kural ile referans arasındaki kappa'yı sınıf dengeli bir alt örneklemde (632 olumsuzun
   tamamı artı sabit bir tohumla çekilmiş 632 olumlu) hesaplayın ve 4c ile karşılaştırın.
   Değişimi [5. bölümün](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/05-veri-etiketleme.html) yaygınlık ve yanlılık paradokslarıyla
   açıklayın, sonra bir rapora hangi sayıyı koyacağınızı söyleyin.

6. **Vahşi doğadaki bir iddia.** 1. bölümün dördüncü alıştırması: son bir ay içinde yapılmış,
   yapay zekâ yetkinliğine dair kamuya açık bir iddia bulun ve altındaki değerlendirmeyi
   tespit edin. Bölümün beşinci mülakat sorusundaki üç soruyu sorun: baseline ne, test
   kümesi ne kadar büyük ve etiketleri kim yazdı?
   [22. bölüm](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/22-llm-degerlendirme.html) değerlendirmenin tam olarak ele alındığı yer.

---

*Bu notebook'taki kod MIT lisanslı; anlatım CC BY 4.0. Türkçe ürün yorumları veri kümesi
CC BY-SA 4.0 (Fatih Barmanbay, Hugging Face'te `fthbrmnby/turkish_product_reviews`).*